Decodificare script 

helper per aprire opcodes.cfg con config parser

In [ ]:
from configparser import ConfigParser

def opcodes(fname):
    config = ConfigParser()
    config.read(fname)
    return {int(config['OPCODES'][code],base=16):code for code in config['OPCODES'] }

if __name__ == '__main__':
    print(opcodes('opcodes.cfg'))


{0: 'op_false', 76: 'op_pushdata1', 77: 'op_pushdata2', 78: 'op_pushdata4', 79: 'op_1negate', 80: 'op_reserved', 81: 'op_true', 82: 'op_2', 83: 'op_3', 84: 'op_4', 85: 'op_5', 86: 'op_6', 87: 'op_7', 88: 'op_8', 89: 'op_9', 90: 'op_10', 91: 'op_11', 92: 'op_12', 93: 'op_13', 94: 'op_14', 95: 'op_15', 96: 'op_16', 97: 'op_nop', 98: 'op_ver', 99: 'op_if', 100: 'op_notif', 101: 'op_verif', 102: 'op_vernotif', 103: 'op_else', 104: 'op_endif', 105: 'op_verify', 106: 'op_return', 107: 'op_toaltstack', 108: 'op_fromaltstack', 109: 'op_2drop', 110: 'op_2dup', 111: 'op_3dup', 112: 'op_2over', 113: 'op_2rot', 114: 'op_2swap', 115: 'op_ifdup', 116: 'op_depth', 117: 'op_drop', 118: 'op_dup', 119: 'op_nip', 120: 'op_over', 121: 'op_pick', 122: 'op_roll', 123: 'op_rot', 124: 'op_swap', 125: 'op_tuck', 126: 'op_cat', 127: 'op_substr', 128: 'op_left', 129: 'op_right', 130: 'op_size', 131: 'op_invert', 132: 'op_and', 133: 'op_or', 134: 'op_xor', 135: 'op_equal', 136: 'op_equalverify', 137: 'op_reserved

script per la serializzazione

In [19]:
class Script:
    
    opcodes = opcodes('opcodes.cfg')
    
    def __init__(self, cmds):
        self.cmds = cmds
    
    @classmethod
    def parse(cls, reader, script_len):
        cmds = []
        counter = 0
        while counter < script_len:
            first = int.from_bytes(reader.read(1), 'little')
            counter += 1
            if 1 <= first <= 75:
                cmds.append(reader.read(first))
                counter += first
            elif first == 76:
                cmd_len = int.from_bytes(reader.read(1), 'little')
                cmds.append(reader.read(cmd_len))
                counter += 1 + cmd_len
            elif first == 77:
                cmd_len = int.from_bytes(reader.read(2), 'little')
                cmds.append(reader.read(cmd_len))
                counter += 2 + cmd_len
            elif first == 78:
                cmd_len = int.from_bytes(reader.read(4), 'little')
                cmds.append(reader.read(cmd_len))
                counter += 4 + cmd_len
                
            else:
                cmds.append(first)    
        return cls(cmds)
    
    def __str__(self):
        return ' '.join([Script.opcodes[cmd] if type(cmd) == int else cmd.hex() for cmd in self.cmds])
    
    
if __name__ == '__main__':
    from io import BytesIO
    scripthex = '410411db93e1dcdb8a016b49840f8c53bc1eb68a382e97b1482ecad7b148a6909a5cb2e0eaddfb84ccf9744464f82e160bfa9b8b64f9d4c03f999b8643f656b412a3ac'
    scripthex = '76a9142c5051b963a80493acddedaa6e851f224157f23988ac'
    script_len = len(bytes.fromhex(scripthex))
    sc = Script.parse(BytesIO(bytes.fromhex(scripthex)), script_len)
    print(sc)

op_dup op_hash160 2c5051b963a80493acddedaa6e851f224157f239 op_equalverify op_checksig


per le Tx mi copio prima quelle utilizzate in block2

In [20]:
import secrets

In [ ]:
class TxOut:
    def __init__(self, value, scriptPubKey):
        self.value = value
        self.scriptPubKey = scriptPubKey
        
    @classmethod
    def parse(cls, reader):
        value = int.from_bytes(reader.read(8), 'little')
        scriptLen = varint2int(reader)
        scriptPubKey = reader.read(scriptLen)
        return cls(value, scriptPubKey)
    
    def __str__(self):
        out = dict()
        out['value'] = self.value
        out['scriptPubKey'] = self.scriptPubKey.hex()
        return out.__str__()
    

In [ ]:
class TxIn:
    def __init__(self, prevTxHash, prevTxOutIndex, scriptSig, sequence):
        self.prevTxHash = prevTxHash
        self.prevTxOutIndex = prevTxOutIndex
        self.scriptSig = scriptSig
        self.sequence = sequence
        
    @classmethod
    def parse(cls, reader):
        prevTxHash = reader.read(32)
        prevTxOutIndex = reader.read(4)
        scriptLen = varint2int(reader)
        scriptSig = reader.read(scriptLen)
        sequence = reader.read(4)
        return cls(prevTxHash, prevTxOutIndex, scriptSig, sequence)
    
    def __str__(self):
        out = dict()
        out['prevTxHash'] = self.prevTxHash.hex()
        out['prevTxOutIndex'] = self.prevTxOutIndex.hex()
        out['scriptSig'] = self.scriptSig.hex()
        out['sequence'] = self.sequence.hex()
        return out.__str__()

In [ ]:
class Tx:
    def __init__(self, version, txIns, txOuts, lockTime):
        self.version = version
        self.txIns = txIns
        self.txOuts = txOuts
        self.lockTime = lockTime
        
    @classmethod
    def parse(cls, reader):
        version = reader.read(4)
        numTxIns = varint2int(reader)
        txIns = [TxIn.parse(reader) for _ in range(numTxIns)]
        numTxOuts = varint2int(reader)
        txOuts = [TxOut.parse(reader) for _ in range(numTxOuts)]
        lockTime = reader.read(4)
        return cls(version, txIns, txOuts, lockTime)
    
    def __str__(self):
        out = dict()
        out['version'] = self.version.hex()
        out['txIns'] = [txIn.__str__() for txIn in self.txIns]
        out['txOuts'] = [txOut.__str__() for txOut in self.txOuts]
        out['lockTime'] = self.lockTime.hex()
        return out.__str__()
    
if __name__ == "__main__":
        txhex = "0100000001c997a5e56e104102fa209c6a852dd90660a20b2d9c352423edce25857fcd3704000000004847304402204e45e16932b8af514961a1d3a1a25fdf3f4f7732e9d624c6c61548ab5fb8cd410220181522ec8eca07de4860a4acdd12909d831cc56cbbac4622082221a8768d1d0901ffffffff0200ca9a3b00000000434104ae1a62fe09c5f51b13905f07f06b99a2f7159b2225f374cd378d71302fa28414e7aab37397f554a7df5f142c21c1b7303b8a0626f1baded5c72a704f7e6cd84cac00286bee0000000043410411db93e1dcdb8a016b49840f8c53bc1eb68a382e97b1482ecad7b148a6909a5cb2e0eaddfb84ccf9744464f82e160bfa9b8b64f9d4c03f999b8643f656b412a3ac00000000"
        reader = BytesIO(bytes.fromhex(txhex))
        tx = Tx.parse(reader)
        print(tx)

{'version': '01000000', 'txIns': ["{'prevTxHash': 'c997a5e56e104102fa209c6a852dd90660a20b2d9c352423edce25857fcd3704', 'prevTxOutIndex': '00000000', 'scriptSig': '47304402204e45e16932b8af514961a1d3a1a25fdf3f4f7732e9d624c6c61548ab5fb8cd410220181522ec8eca07de4860a4acdd12909d831cc56cbbac4622082221a8768d1d0901', 'sequence': 'ffffffff'}"], 'txOuts': ["{'value': 1000000000, 'scriptPubKey': '4104ae1a62fe09c5f51b13905f07f06b99a2f7159b2225f374cd378d71302fa28414e7aab37397f554a7df5f142c21c1b7303b8a0626f1baded5c72a704f7e6cd84cac'}", "{'value': 4000000000, 'scriptPubKey': '410411db93e1dcdb8a016b49840f8c53bc1eb68a382e97b1482ecad7b148a6909a5cb2e0eaddfb84ccf9744464f82e160bfa9b8b64f9d4c03f999b8643f656b412a3ac'}"], 'lockTime': '00000000'}
